# 03 — Fit3D: offline vs streaming features (PC)

Uses the MediaPipe cache that notebook 01 already wrote (`data/fit3d/cache`), so no
video is processed again. For every arm-recording it builds the features twice:

* **offline:** the whole recording at once (what the model was trained on so far);
* **streaming:** frame by frame through `iml_live.stream_features()`, each row using only earlier frames (what the live system computes).

Same truth, same lag, same gates as notebook 01. Output: `data/fit3d/feat_fit3d_stream.npz`
(git-ignored; Fit3D licence).

In [ ]:
# ---------------- settings: edit these ----------------
import os, sys, glob, json, time
# Windows PC example (raw strings r'...' because of the backslashes and spaces)
REPO       = r'F:\namal sir project\github repo\AI-based-Physiotherapy-project-'              # <-- your repo folder
FIT3D_ROOT = r'F:\namal sir project\fit3d dataset\curls\train'        # folder holding s03, s04, ...
# ADA / Linux example:
# REPO = os.path.expanduser('~/AI-based-Physiotherapy-project-'); FIT3D_ROOT = os.path.expanduser('~/fit3d/train')
OUT        = os.path.join(REPO, 'data', 'fit3d')                       # git-ignored
EXERCISES  = ['dumbbell_biceps_curls', 'dumbbell_hammer_curls', 'drag_curl']
EXTRACT_ONLY = False     # True = stop after MediaPipe (e.g. on your laptop), then copy data/fit3d/cache
STEP       = 2          # every 2nd frame: 50 fps -> 25 fps (close to a 30 fps webcam)
N_WORKERS  = max(1, (os.cpu_count() or 2) - 1)
THRESHOLD, MAX_LAG, MIN_ROM, MIN_COV = 0.70, 30, 60.0, 50.0   # same as the REHAB24-6 benchmark

def find_repo(path):
    '''use the path given; if it is wrong, look upwards from the notebook folder'''
    cands = [path, os.getcwd(), os.path.dirname(os.getcwd())]
    for c in cands:
        if os.path.exists(os.path.join(c, 'calibration.py')) and os.path.exists(os.path.join(c, 'pose', 'landmarks.py')):
            if c != path:
                print('REPO path not found as typed; using', c)
            return os.path.abspath(c)
    raise FileNotFoundError(f'repo not found at {path!r} - set REPO to the folder that holds calibration.py')
REPO = find_repo(REPO)
OUT = os.path.join(REPO, 'data', 'fit3d')
assert os.path.isdir(FIT3D_ROOT), f'FIT3D_ROOT not found: {FIT3D_ROOT}'
os.makedirs(os.path.join(OUT, 'cache'), exist_ok=True)
sys.path[:0] = [REPO, os.path.join(REPO, 'pose'), os.path.join(REPO, 'scripts')]
print('subjects:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(FIT3D_ROOT, 's*'))))
print('workers :', N_WORKERS)

In [ ]:
# ---------------- repo code (identical to the benchmark) ----------------
import inspect
import numpy as np, pandas as pd
import angles as angles_2d
from angles import elbow_angle_3d_ordered
from angles_lift import bone_lengths, elbow_angle_lifted
from angles_world import mask_world_like
from visibility import mask_low_visibility
from features import frame_features, overlap_rows, FEATURE_NAMES
import benchmark_common as bc

assert 'lfilter_zi' in inspect.getsource(bc.butterworth_filter), \
    'benchmark_common.butterworth_filter is not causal yet - run cell A of notebook 00 first'
CAUSAL = True

def fill_1d(arr, max_gap=5):
    '''Copied from the benchmark notebook (STEP 5) so both datasets are treated alike.'''
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(idx[gap_start:i], idx[good], arr[good])
    return filled

def smooth(x, fps):
    x = fill_1d(x, 5)
    return bc.butterworth_filter(x, fs=fps, cutoff=6.0, causal=CAUSAL) if np.isfinite(x).sum() > 15 else x
from iml_live import stream_features

## 1 — Recordings in the cache

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mproc

def cache_path(subj, cam, ex):
    return os.path.join(OUT, 'cache', f'{subj}_{cam}_{ex}_s{STEP}_w.npz')

def extract_one(job):
    video, cache = job
    if os.path.exists(cache):
        return cache, 'cached', 0.0
    import cv2
    try:
        from pose.landmarks import LandmarkExtractor
    except ImportError:
        from pose.landmarks import LandmarkExtractor
    t0 = time.time()
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 50.0
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ext = LandmarkExtractor(model_complexity=1)
    lms, wds, frames, i = [], [], [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % STEP == 0:
            n, w = ext.process_with_world(frame)
            lms.append(n); wds.append(w); frames.append(i)
        i += 1
    cap.release(); ext.close()
    np.savez_compressed(cache, lm=np.asarray(lms, np.float32), world=np.asarray(wds, np.float32),
                        frames=np.asarray(frames), size=np.asarray([W, H]), fps=fps / STEP)
    return cache, 'new', time.time() - t0

# Recordings are found from the camera files, so the videos are only needed for
# recordings that are not cached yet (you can run MediaPipe on your laptop and
# copy only data/fit3d/cache/ to the server).
jobs, index, missing = [], [], []
for sd in sorted(glob.glob(os.path.join(FIT3D_ROOT, 's*'))):
    subj = os.path.basename(sd)
    for ex in EXERCISES:
        for cp in sorted(glob.glob(os.path.join(sd, 'camera_parameters', '*', ex + '.json'))):
            cam = os.path.basename(os.path.dirname(cp))
            v, c = os.path.join(sd, 'videos', cam, ex + '.mp4'), cache_path(subj, cam, ex)
            if os.path.exists(c) or os.path.exists(v):
                jobs.append((v, c)); index.append((subj, cam, ex))
            else:
                missing.append(f'{subj}/{cam}/{ex}')
print(len(jobs), 'recordings;', sum(os.path.exists(c) for _, c in jobs), 'already cached;',
      len(missing), 'with neither video nor cache (skipped)')

todo = [j for j in jobs if not os.path.exists(j[1])]
t0 = time.time()
if todo and 'fork' in mproc.get_all_start_methods() and N_WORKERS > 1:      # Linux / ADA
    with ProcessPoolExecutor(N_WORKERS, mp_context=mproc.get_context('fork')) as pool:
        for k, (c, how, sec) in enumerate(pool.map(extract_one, todo), 1):
            print(f'[{k}/{len(todo)}] {os.path.basename(c)}  {sec:.0f}s', flush=True)
else:                                     # Windows: no fork in notebooks -> one by one
    for k, job in enumerate(todo, 1):
        c, how, sec = extract_one(job)
        print(f'[{k}/{len(todo)}] {os.path.basename(c)}  {sec:.0f}s', flush=True)
# stopped half way? just run this cell again - finished videos are cached and skipped
print(f'done in {(time.time() - t0) / 60:.1f} min')
if EXTRACT_ONLY:
    raise SystemExit('EXTRACT_ONLY: cache written to ' + os.path.join(OUT, 'cache') + ' - copy that folder to the server')

## 2 — Which Fit3D joints are shoulder / elbow / wrist? (same as notebook 01)

In [ ]:
MP = {'left': (11, 13, 15), 'right': (12, 14, 16)}

def load_json(p):
    with open(p, encoding='utf-8') as f:
        return json.load(f)

def load_joints(p):
    d = load_json(p)
    a = np.asarray(d.get('joints3d_25', d) if isinstance(d, dict) else d, float)
    assert a.ndim == 3 and a.shape[1:] == (25, 3), a.shape
    return a

def load_camera(p):
    d = load_json(p)
    ex = d['extrinsics']
    it = d.get('intrinsics_wo_distortion') or d.get('intrinsics_w_distortion') or d.get('intrinsics')
    return (np.asarray(ex['R'], float).reshape(3, 3), np.asarray(ex['T'], float).reshape(3),
            np.asarray(it['f'], float).reshape(-1), np.asarray(it['c'], float).reshape(-1))

def to_camera(X, R, T, mode):
    return [(X - T) @ R.T, X @ R.T + T, (X - T) @ R, X @ R + T][mode]

def project(Xc, f, c):
    with np.errstate(invalid='ignore', divide='ignore'):
        return Xc[..., :2] / Xc[..., 2:3] * f + c

def match(J, lm_px, R, T, f, c):
    best = None
    for mode in range(4):
        Xc = to_camera(J, R, T, mode)
        if np.nanmedian(Xc[..., 2]) <= 0:
            continue
        P = project(Xc, f, c)
        mapping, errs = {}, []
        for side, idx in MP.items():
            pick = []
            for m in idx:
                med = np.nanmedian(np.linalg.norm(P - lm_px[:, m:m + 1, :2], axis=-1), axis=0)
                j = int(np.nanargmin(med)); pick.append(j); errs.append(med[j])
            mapping[side] = tuple(pick)
        e = float(np.nanmedian(errs))
        if best is None or e < best[2]:
            best = (mode, mapping, e)
    return best

def load_rec(subj, cam, ex):
    sd = os.path.join(FIT3D_ROOT, subj)
    c = np.load(cache_path(subj, cam, ex))
    joints = load_joints(os.path.join(sd, 'joints3d_25', ex + '.json'))
    frames = c['frames']; keep = frames < len(joints)
    return dict(lm=c['lm'][keep].astype(float), world=c['world'][keep].astype(float),
                frames=frames[keep], size=tuple(int(v) for v in c['size']), fps=float(c['fps']),
                J=joints[frames[keep]],
                cam=load_camera(os.path.join(sd, 'camera_parameters', cam, ex + '.json')))

print(len(index), 'recordings to check')
if not index:
    raise RuntimeError('Section 1 found no videos. Check FIT3D_ROOT (it must hold s03, s04, ...) '
                       'and that section 1 printed "N videos" with N > 0.')
votes, problems = [], []
for subj, cam, ex in index:
    try:
        r = load_rec(subj, cam, ex)
    except Exception as e:
        problems.append((subj, cam, ex, f'{type(e).__name__}: {e}')); continue
    seen = np.isfinite(r['lm'][:, [11, 12, 13, 14, 15, 16], 0]).all(axis=1)
    if seen.sum() < 20:
        problems.append((subj, cam, ex, f'MediaPipe found the arms in only {int(seen.sum())} frames')); continue
    px = r['lm'][seen].copy(); px[..., 0] *= r['size'][0]; px[..., 1] *= r['size'][1]
    res = match(r['J'][seen][::5], px[::5], *r['cam'])
    if res is None:
        problems.append((subj, cam, ex, 'joints project behind the camera in every convention')); continue
    mode, mapping, err = res
    votes.append((subj, cam, ex, mode, str(mapping['left']), str(mapping['right']), round(err, 1)))

if problems:
    print('skipped while matching:')
    for p in problems[:20]:
        print('  ', *p)
V = pd.DataFrame(votes, columns=['subject', 'camera', 'exercise', 'mode', 'left', 'right', 'px_err'])
if V.empty:
    raise RuntimeError('No recording could be matched - send the "skipped while matching" lines above.')
key = V.groupby(['mode', 'left', 'right']).size().sort_values(ascending=False)
print(key.head())
(MODE, LEFT, RIGHT), n_agree = key.index[0], int(key.iloc[0])
MODE, LEFT, RIGHT = int(MODE), tuple(int(v) for v in LEFT.strip('()').split(',')), \
                    tuple(int(v) for v in RIGHT.strip('()').split(','))
JMAP = {'left': LEFT, 'right': RIGHT}
good = V[(V['mode'] == MODE) & (V['left'] == str(LEFT)) & (V['right'] == str(RIGHT))]
print(f'\nusing mode {MODE}, left {LEFT}, right {RIGHT}  ({n_agree}/{len(V)} recordings agree; '
      f'median px error {good.px_err.median():.1f})')
if n_agree < 0.6 * len(V) or good.px_err.median() > 40:
    print('WARNING: weak agreement or large pixel error - send this table before trusting the results')

## 3 — Offline and streaming features for every arm

In [ ]:
def view_of(J, R, T):
    Xc = to_camera(J, R, T, MODE)
    v = Xc[:, JMAP['left'][0]] - Xc[:, JMAP['right'][0]]
    yaw = float(np.nanmedian(np.degrees(np.arctan2(np.abs(v[:, 2]), np.linalg.norm(v[:, :2], axis=1)))))
    return yaw, 'front' if yaw < 30 else 'half' if yaw < 60 else 'profile'

I2D = FEATURE_NAMES.index('angle_2d')
rows, parts, skipped = [], [], {'static': 0, 'occluded': 0, 'align': 0}
t0 = time.time()
for k, (subj, cam, ex) in enumerate(index, 1):
    r = load_rec(subj, cam, ex)
    lm, world, size, fps = r['lm'], r['world'], r['size'], r['fps']
    R, T, f, c = r['cam']
    yaw, view = view_of(r['J'], R, T)
    lm_m = mask_low_visibility(lm, threshold=THRESHOLD)
    wd_m = mask_world_like(world, lm_m)
    for side in ('right', 'left'):
        s, e, w = JMAP[side]
        gt = angles_2d.angle(r['J'][:, s], r['J'][:, e], r['J'][:, w])
        L_up, L_fo = bone_lengths(lm_m, side, size)
        pa = smooth(angles_2d.elbow_angle(lm_m, side=side, size=size), fps)
        pb = smooth(elbow_angle_3d_ordered(wd_m, side=side), fps)
        pe = smooth(elbow_angle_lifted(lm_m, wd_m, side, size, L_up, L_fo), fps)
        feat = frame_features(lm_m, wd_m, side, size, L_up, L_fo, pa, pb, pe)

        rom = float(np.nanpercentile(gt, 95) - np.nanpercentile(gt, 5))
        cov = 100.0 * float(np.isfinite(pa).mean())
        if rom < MIN_ROM:
            skipped['static'] += 1; continue
        if cov < MIN_COV:
            skipped['occluded'] += 1; continue
        lag, lag_r = bc.best_lag(pa, gt, max_lag=MAX_LAG, filter_signals=True, fs=fps)
        if abs(lag) >= MAX_LAG or lag_r < 0.7:
            skipped['align'] += 1; continue

        stream = stream_features(lm, world, side, size, fps, threshold=THRESHOLD)   # frame by frame
        F, g = overlap_rows(feat, gt, lag)
        S, _ = overlap_rows(stream, gt, lag)
        ok = np.isfinite(g) & (np.isfinite(F[:, I2D]) | np.isfinite(S[:, I2D]))
        rec = f'fit3d_{subj}_{cam}_{ex}_{side}'
        parts.append((F[ok], S[ok], g[ok], subj, rec, view, ex))
        both = ok & np.isfinite(F[:, I2D]) & np.isfinite(S[:, I2D])
        rows.append(dict(recording=rec, view=view, lag=lag, frames=int(ok.sum()),
                         angle2d_diff=round(float(np.nanmedian(np.abs(F[both, I2D] - S[both, I2D]))), 3)))
    print(f'[{k}/{len(index)}] {subj} {cam} {ex}  {time.time() - t0:.0f}s', flush=True)

REC = pd.DataFrame(rows)
print('kept', len(REC), 'arm-recordings; skipped', skipped)
REC.groupby('view').agg(recordings=('recording', 'size'), frames=('frames', 'sum'),
                        angle2d_offline_vs_stream_deg=('angle2d_diff', 'median'))

In [ ]:
out = os.path.join(OUT, 'feat_fit3d_stream.npz')
n = [len(p[2]) for p in parts]
np.savez_compressed(
    out,
    X_offline=np.concatenate([p[0] for p in parts]).astype(np.float32),
    X_stream=np.concatenate([p[1] for p in parts]).astype(np.float32),
    y=np.concatenate([p[2] for p in parts]).astype(np.float32),
    feature_names=np.array(FEATURE_NAMES),
    person=np.repeat([f'fit3d_{p[3]}' for p in parts], n),
    recording=np.repeat([p[4] for p in parts], n),
    view=np.repeat([p[5] for p in parts], n),
    exercise=np.repeat([p[6] for p in parts], n),
    dataset=np.array(['fit3d'] * sum(n)),
    causal=np.array(True),
)
print(f'saved {out}: {sum(n):,} frames from {len(parts)} arm-recordings')